# Pydantic Basics

*Notebook 06*

Pydantic is a Python library for defining typed data models that validate themselves. We use it throughout the course because it catches bad agent output at the boundary instead of letting malformed data propagate downstream. In Lesson 07 we'll use it to force agents to return structured output — but first, this notebook covers the basics so you can read and write Pydantic models with confidence.

**Topics:**
- Defining a model with `BaseModel` and typed fields

- Valid vs invalid data — seeing `ValidationError` in action

- Field constraints with `Field` and `Annotated`

- Accessing fields with dot notation

- Serializing models with `model_dump()` and `model_dump_json()`

- Why this course uses Pydantic instead of raw JSON schemas

---

## 🔧 Setup

In [ ]:
from typing import Annotated
from pydantic import BaseModel, Field, ValidationError

print("✅ Ready!")

---

## 📋 Part 1: Your First Pydantic Model

A Pydantic model is a Python class that inherits from `BaseModel` and uses type hints to define its fields.

In [ ]:
class Person(BaseModel):
    name: str
    age: int
    email: str

# --------------------------------------------------------------
print("✅ Person model ready")

Creating an instance works like any Python class — but Pydantic validates the types automatically.

In [ ]:
person = Person(name="Alice", age=30, email="alice@example.com")

print(person)

---

## ⚠️ Part 2: Validation in Action

If the data doesn't match the model, Pydantic raises a `ValidationError` with a clear explanation of what went wrong. In agent workflows, this is the safety net that catches malformed output before your next tool call or downstream code tries to use it.

#### Invalid: Wrong Type

In [ ]:
try:
    person = Person(name="Bob", age="thirty", email="bob@example.com")
except ValidationError as e:
    print(e)

#### Invalid: Missing Field

In [ ]:
try:
    person = Person(name="Carol", age=25)
except ValidationError as e:
    print(e)

---

## 📏 Part 3: Field Constraints

Type hints alone handle the basics — string, int, bool. For tighter rules (a rating between 1 and 5, a name no longer than 50 characters), use `Field` with `Annotated`. `Annotated` lets you attach extra metadata — here, the `Field` constraints — to a type hint without changing the type itself. `ge` and `le` mean *greater-or-equal* and *less-or-equal*.

In [ ]:
class ProductReview(BaseModel):
    product_name: Annotated[str, Field(max_length=50)]
    rating: Annotated[int, Field(ge=1, le=5)]
    comment: Annotated[str, Field(max_length=200)]

# --------------------------------------------------------------
print("✅ ProductReview model ready")

#### Valid Review

In [ ]:
review = ProductReview(
    product_name="Wireless Headphones",
    rating=4,
    comment="Great sound quality, comfortable fit."
)

print(review)

#### Invalid: Rating Out of Range

In [ ]:
try:
    review = ProductReview(
        product_name="Wireless Headphones",
        rating=7,
        comment="Amazing!"
    )
except ValidationError as e:
    print(e)

### 💡 Why This Works

Constraints live in the model, not in the prompt. These constraints become useful in Lesson 07, where agent output is validated against the model — there's no need to ask the agent to "please return a rating between 1 and 5" (and hope it listens).

---

## 🔍 Part 4: Accessing and Serializing Models

Once a model instance is created, access fields with standard dot notation.

In [ ]:
review = ProductReview(
    product_name="Laptop Stand",
    rating=5,
    comment="Sturdy and adjustable."
)

print(f"Product: {review.product_name}")
print(f"Rating: {review.rating}/5")
print(f"Comment: {review.comment}")

Pydantic models also serialize cleanly to dictionaries and JSON. You'll use this any time validated output needs to leave the model — logging an agent result, sending it to another API, or saving it to disk. Use `model_dump()` when passing data to other Python code; use `model_dump_json()` when sending output to an API or saving to a file.

In [ ]:
print("As dict:")
print(review.model_dump())

print("\nAs JSON:")
print(review.model_dump_json(indent=2))

---

## 🧠 Why This Course Uses Pydantic

The Agents SDK can accept either Pydantic models *or* hand-written JSON schemas for output validation. This course uses Pydantic for three reasons:

- **Validation is automatic** — if the agent returns bad data, you know immediately instead of crashing three steps downstream
- **Type hints are readable** — a Pydantic class is easier to read and maintain than a hand-written JSON schema
- **IDE support** — autocomplete on `review.rating` works; autocomplete on `data["rating"]` does not

---

## 💪 Practice Exercises

### Exercise 1: Book Model

*Covers: `BaseModel` — defining and testing a typed Pydantic model*

Define a Pydantic `Book` model and test it with valid and invalid data.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Book Model
# --------------------------------------------------------------
# Objective: Define a Pydantic model and test valid and invalid data.

# TODO 1: Define a Book model with:
#            - title (str)
#            - author (str)
#            - year (int)
#            - pages (int)

# TODO 2: Create a valid Book instance and print it

# TODO 3: Try to create an invalid Book (e.g., year as a string)
#            Catch the ValidationError and print the message

# --- Write your code below this line ---

### Exercise 2: Constrained Survey Response

*Covers: `Field` constraints — bounded values and length limits*

Build a model for a customer survey with a bounded satisfaction score and a length-limited comment.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Constrained Survey Response
# --------------------------------------------------------------
# Objective: Use Field constraints to enforce validation rules.

# TODO 1: Define a SurveyResponse model with:
#            - customer_name (str, max 100 chars)
#            - satisfaction (int, 1-10)
#            - feedback (str, max 500 chars)
#            - Hint: use Annotated[int, Field(ge=1, le=10)] for satisfaction

# TODO 2: Create a valid SurveyResponse and print it

# TODO 3: Try to create one with satisfaction=15
#            Catch the ValidationError and print the message

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Pydantic models define typed, self-validating data:**

- Inherit from `BaseModel` and use type hints for each field
- Pydantic checks types automatically at construction time
- Invalid data raises `ValidationError` with clear diagnostics
<br>
<br>

**Field constraints live in the model:**

- Use `Annotated[type, Field(...)]` for bounded values and length limits
- Common constraints: `ge`, `le`, `min_length`, `max_length`
- Constraints are enforced the same way as type checks
<br>
<br>

**Pydantic over raw JSON schemas:**

- Automatic validation, readable syntax, IDE autocomplete
- Models serialize cleanly with `model_dump()` and `model_dump_json()`

---

## 📍 Next Step

07: Structured Outputs — Use Pydantic models with the Agents SDK to force agents to return typed, validated data.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-06-pydantic-basics)

---